(what-is-ecological-economics)=
# What Is Ecological Economics?

Mainstream (neoclassical) economics usually treats "the economy" as the whole system: households, firms, and markets exchanging goods and services, with nature entering mostly as a source of raw materials or as an occasional "externality" to be priced in when something goes wrong (like pollution).

**Ecological economics (EE)** turns that picture inside out. It starts from the observation that the economy is a *subsystem* embedded within a larger, finite, non-growing biosphere. Everything the economy produces or consumes ultimately draws down energy and material stocks and returns waste and heat back into that same biosphere. If you take that seriously, the question is no longer just "how do we allocate scarce resources efficiently?" but also "how large can the economic subsystem physically become, and what happens as it approaches the scale of the whole system that contains it?"

**A concrete sense of scale:** the [Python intro chapter](../intro_python/calculator.ipynb) of this book already works with global energy figures — about **592 EJ** of primary energy consumed per year by roughly **8.2 billion people**, or about **72 GJ per person per year**. That entire flow — everything it takes to power farms, factories, transport, and homes for the whole human economy — is the "economic subsystem" this chapter is talking about. It has to come from somewhere (the biosphere's energy and material stocks) and it has to go somewhere (waste heat, emissions, degraded materials, back into that same biosphere).

## A biophysical starting point

This way of thinking is closely tied to physics and thermodynamics rather than to economics alone. Some of the field's founding figures were mathematicians and physicists by training.

### Georgescu-Roegen: entropy is a one-way street

[Nicholas Georgescu-Roegen](https://en.wikipedia.org/wiki/Nicholas_Georgescu-Roegen) argued that economic production is subject to the laws of thermodynamics — in particular the second law (entropy) — meaning that material and energy transformations are ultimately irreversible, not a closed circular flow.

The toy model below makes this concrete. Start with 100 units of a usable ("low-entropy") resource. Each cycle, part of it is used, and only a fraction of that use is ever recovered through recycling — the rest becomes unusable ("high-entropy") waste. Try sliding recycling efficiency all the way up to 95% below: the usable stock still declines every single cycle. Recycling can slow the entropy increase down a lot, but it can never reverse it.

In [1]:
import plotly.graph_objects as go
from IPython.display import HTML


def entropy_depletion(stock0=100.0, use_rate=0.10, recycling_efficiency=0.0, cycles=30):
    stock = stock0
    waste = 0.0
    stocks = [stock]
    wastes = [waste]
    for _ in range(cycles):
        used = stock * use_rate
        recovered = used * recycling_efficiency
        lost = used - recovered
        stock = stock - lost
        waste += lost
        stocks.append(stock)
        wastes.append(waste)
    return stocks, wastes


recycling_levels = [0.0, 0.25, 0.5, 0.75, 0.95]
cycles_axis = list(range(31))

fig = go.Figure()
for i, eff in enumerate(recycling_levels):
    stocks, wastes = entropy_depletion(recycling_efficiency=eff)
    fig.add_trace(go.Scatter(x=cycles_axis, y=stocks, mode="lines", name="Usable stock",
                              line=dict(color="#2E7D32", width=3), visible=(i == 0)))
    fig.add_trace(go.Scatter(x=cycles_axis, y=wastes, mode="lines", name="Cumulative waste",
                              line=dict(color="#B71C1C", width=3, dash="dash"), visible=(i == 0)))

steps = []
for i, eff in enumerate(recycling_levels):
    visible = [False] * (2 * len(recycling_levels))
    visible[2 * i] = True
    visible[2 * i + 1] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{eff:.0%}"))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Recycling efficiency: "}, steps=steps)],
    title="Even near-perfect recycling only slows entropy increase \u2014 it never reverses it",
    xaxis_title="Cycle (e.g. years)",
    yaxis_title="Units (usable stock vs. cumulative waste)",
    height=450,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

**What to notice:** even at 95% recycling efficiency the usable stock still ends up lower than where it started after 30 cycles — compare that to 0% recycling, where the stock collapses to about 4% of its starting value. Recycling buys time; it does not create a closed loop.

### Ayres: economies have a metabolism

[Robert Ayres](https://en.wikipedia.org/wiki/Robert_Ayres_(scientist)) developed *industrial metabolism* and material/energy flow accounting, treating economies the way an ecologist treats an organism's metabolism: you cannot sustain a given level of activity without a matching flow of energy and materials through the system.

The chart below uses real 2023 cross-country data (the same file used later in the [static scaling chapter](../order_of_magnitude_economics_and_scaling/static_scale.ipynb)): energy consumption per capita against GDP per capita, on a log-log scale. Slide through different target GDP-per-capita levels and read off the energy throughput the fitted trend line associates with sustaining that level of economic activity.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML

df = pd.read_csv("../../teaching_data_sets/data2_energygdp_owid2023.csv", sep=";")
df = df.dropna(subset=["energy_pc", "gdp_pc"])
df = df[(df["energy_pc"] > 0) & (df["gdp_pc"] > 0)]

log_gdp = np.log10(df["gdp_pc"])
log_energy = np.log10(df["energy_pc"])
slope, intercept = np.polyfit(log_gdp, log_energy, 1)


def predict_energy(gdp_pc):
    return 10 ** (slope * np.log10(gdp_pc) + intercept)


target_gdps = [2000, 8000, 20000, 50000, 100000]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["gdp_pc"], y=df["energy_pc"], mode="markers",
    marker=dict(size=7, color="#1565C0", opacity=0.6),
    name="Countries (2023)",
))
x_line = np.logspace(np.log10(df["gdp_pc"].min()), np.log10(df["gdp_pc"].max()), 100)
fig.add_trace(go.Scatter(
    x=x_line, y=predict_energy(x_line), mode="lines",
    line=dict(color="#EF6C00", width=2), name="Fitted trend",
))

for i, target in enumerate(target_gdps):
    predicted = predict_energy(target)
    fig.add_trace(go.Scatter(
        x=[target, target], y=[df["energy_pc"].min(), predicted],
        mode="lines", line=dict(color="#6A1B9A", width=2, dash="dot"),
        visible=(i == 0), name="Target GDP per capita", showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=[target], y=[predicted], mode="markers+text",
        marker=dict(size=12, color="#6A1B9A", symbol="diamond"),
        text=[f"{predicted:,.0f} kWh/person"], textposition="top center",
        visible=(i == 0), name="Predicted energy throughput", showlegend=False,
    ))

n_base = 2
steps = []
for i, target in enumerate(target_gdps):
    visible = [True, True] + [False] * (2 * len(target_gdps))
    visible[n_base + 2 * i] = True
    visible[n_base + 2 * i + 1] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"${target:,}"))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Target GDP per capita: "}, steps=steps)],
    xaxis=dict(type="log", title="GDP per capita ($)"),
    yaxis=dict(type="log", title="Energy consumption per capita (kWh)"),
    title="Sustaining a given GDP requires a matching energy ‘metabolism’ (2023 cross-country data)",
    height=500,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

**What to notice:** the fitted relationship has a slope of about 1.2 in log-log space — energy use per capita rises slightly *faster* than proportionally with GDP per capita across these 189 countries. There is no country in this data sustaining a high-GDP lifestyle on a low energy throughput.

### Daly: growth economy vs. steady-state economy

[Herman Daly](https://en.wikipedia.org/wiki/Herman_Daly) formalized the idea of a **steady-state economy**: an economy whose physical throughput of matter and energy stays within ecological limits, even while qualitative development continues.

The chart below plots real global energy-consumption-per-capita data from 1965–2024 (World aggregate) against a few hypothetical steady-state proposals — pick a scenario from the slider to see how different a capped trajectory looks from the actual growth path.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML

df = pd.read_csv("../../teaching_data_sets/data1_energygdp_owid.csv", on_bad_lines="skip")
world = df[df["Entity"] == "World"].sort_values("Year")
years = world["Year"].tolist()
energy = world["Per capita energy consumption"].tolist()

steady_state_options = [
    {"level": energy[years.index(1995)], "start_year": 1995, "label": "Cap at 1995 level (~17,600 kWh)"},
    {"level": 16000, "start_year": 1995, "label": "Cap at 16,000 kWh, starting 1995"},
    {"level": 16000, "start_year": 2010, "label": "Cap at 16,000 kWh, starting 2010"},
    {"level": 18000, "start_year": 2010, "label": "Cap at 18,000 kWh, starting 2010"},
]


def steady_state_path(level, start_year):
    return [e if y < start_year else level for y, e in zip(years, energy)]


fig = go.Figure()
fig.add_trace(go.Scatter(x=years, y=energy, mode="lines", name="Actual growth trajectory",
                          line=dict(color="#B71C1C", width=3)))

for i, opt in enumerate(steady_state_options):
    path = steady_state_path(opt["level"], opt["start_year"])
    fig.add_trace(go.Scatter(x=years, y=path, mode="lines", name="Steady-state proposal",
                              line=dict(color="#2E7D32", width=3, dash="dash"),
                              visible=(i == 0), showlegend=(i == 0)))

steps = []
for i, opt in enumerate(steady_state_options):
    visible = [True] + [False] * len(steady_state_options)
    visible[1 + i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=opt["label"]))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Steady-state scenario: "}, steps=steps)],
    xaxis_title="Year", yaxis_title="World primary energy consumption per capita (kWh)",
    title="Growth economy vs. Daly’s steady-state proposal (real global data, 1965-2024)",
    height=450,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

**What to notice:** none of the steady-state scenarios below claim the world ever actually adopted one — they are counterfactuals. The point is structural: a steady-state economy is defined by *capping physical throughput*, not by freezing wellbeing or technological progress.

### Boulding: cowboy economy vs. spaceman economy

[Kenneth Boulding](https://en.wikipedia.org/wiki/Kenneth_Boulding) captured the intuition memorably by contrasting the old "cowboy economy" (endless open frontier, waste is someone else's problem) with the "spaceman economy" (a closed system where inputs and waste sinks are both finite).

The simulation below starts both economies with the same 1,000-unit resource stock. The "cowboy" economy extracts a fixed amount every year no matter how much is left, drawing down a stock that never replenishes. The "spaceman" economy extracts that same fixed amount, but from a stock that regenerates by 5% per year (up to its original size) — a renewable resource instead of a non-renewable one. Slide the extraction rate up and watch what happens to each.

In [4]:
import plotly.graph_objects as go
from IPython.display import HTML


def simulate_boulding(stock0=1000.0, regen_rate=0.05, extraction_rate=0.05, years=40):
    cowboy = stock0
    spaceman = stock0
    cowboy_path = [cowboy]
    spaceman_path = [spaceman]
    fixed_harvest = extraction_rate * stock0
    for _ in range(years):
        cowboy = max(0.0, cowboy - fixed_harvest)
        harvest_s = min(extraction_rate * stock0, spaceman)
        spaceman = min(stock0, (spaceman - harvest_s) * (1 + regen_rate))
        cowboy_path.append(cowboy)
        spaceman_path.append(spaceman)
    return cowboy_path, spaceman_path


extraction_levels = [0.02, 0.04, 0.06, 0.08, 0.10]
years_axis = list(range(41))

fig = go.Figure()
for i, rate in enumerate(extraction_levels):
    cowboy_path, spaceman_path = simulate_boulding(extraction_rate=rate)
    fig.add_trace(go.Scatter(x=years_axis, y=cowboy_path, mode="lines", name="Cowboy economy",
                              line=dict(color="#B71C1C", width=3), visible=(i == 0)))
    fig.add_trace(go.Scatter(x=years_axis, y=spaceman_path, mode="lines", name="Spaceman economy",
                              line=dict(color="#2E7D32", width=3), visible=(i == 0)))

steps = []
for i, rate in enumerate(extraction_levels):
    visible = [False] * (2 * len(extraction_levels))
    visible[2 * i] = True
    visible[2 * i + 1] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{rate:.0%} of stock/year"))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Extraction rate: "}, steps=steps)],
    xaxis_title="Year", yaxis_title="Resource stock remaining (units)",
    title="Cowboy vs. spaceman economy \u2014 extraction rate vs. regeneration capacity (5%/year)",
    height=450,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

**What to notice:** at low extraction rates, the spaceman economy stabilizes while the cowboy economy still runs its stock down toward zero. But push the extraction rate above the 5%/year regeneration rate, and the spaceman economy collapses too — respecting *some* limit is not automatically enough; it has to be the right limit.

## How this differs from environmental economics

Ecological economics is often confused with *environmental economics*, and the two overlap heavily in practice, but they start from different premises:

- **Environmental economics** largely stays inside the neoclassical framework: nature's services are assigned prices (or shadow prices), and environmental problems are treated as market failures to be corrected — usually under an assumption of **weak sustainability**, where natural capital can in principle be substituted by produced (human-made) capital, as long as total capital does not decline.
- **Ecological economics** more often leans toward **strong sustainability**: certain forms of natural capital (a stable climate, functioning ecosystems, biodiversity) are treated as non-substitutable — no amount of built capital can fully replace them.

### A worked example: draining a wetland

Say a country converts a wetland into industrial land. The wetland was providing an estimated $50 million per year in flood-protection value (by absorbing storm surge), plus harder-to-price benefits: habitat, water filtration, carbon storage. The new factories built on the drained land are worth $200 million.

- **Weak sustainability view:** total capital rose — natural capital worth roughly $50M/year-equivalent was traded for produced capital worth $200M. As long as some of that $200M is reinvested (e.g., in levees that replace the lost flood protection), total capital is maintained or increased, so this is acceptable.
- **Strong sustainability view:** no levee replicates *all* of what the wetland did simultaneously — flood buffering *and* habitat *and* filtration *and* carbon storage. Some of that natural capital was uniquely valuable and is now irreversibly gone, regardless of the monetary capital gain.

This book, in line with the [Preface](../preface.md), stays mostly descriptive and quantitative rather than staking out a strong normative position — but understanding this distinction is essential background for reading the rest of the field.

## What ecological economists actually do

In practice, the field builds and uses quantitative tools to describe the economy-environment system rather than treating the two as separate:

- **Scaling and order-of-magnitude reasoning** — recognizing that variables like GDP, energy use, or emissions span many orders of magnitude across countries and time. The {ref}`next chapter <order_of_magnitude_economics_and_scaling>` builds this intuition directly, and you already saw it in action above: GDP per capita in the Ayres chart spans from about \$1,000 to \$130,000 across countries — roughly two orders of magnitude.
- **Biophysical accounting** — energy and material flow analysis, footprint accounting, input-output analysis that tracks physical flows alongside monetary ones, exactly like the Ayres and Daly charts above.
- **Modelling coupled systems** — agent-based models and system dynamics models that let population, resource stocks, technology, and economic output co-evolve, rather than assuming a fixed environment, in the spirit of the Boulding simulation above (a dedicated chapter on these modelling techniques is planned for later in the book).

These are exactly the kinds of tools this book will build up, chapter by chapter, using Python as the computational backbone (see {ref}`intro-python`).

:::{tip}
**What you'll learn in this chapter**
- What distinguishes ecological economics from mainstream (neoclassical) and environmental economics
- The biophysical/thermodynamic starting point of the field (Georgescu-Roegen, Ayres, Daly, Boulding), made concrete with interactive models you can play with
- The difference between weak and strong sustainability, with a worked example
- What kinds of quantitative tools ecological economists use, and how they map onto the rest of this book
:::